# Snowflake x GenAI: LLM Functions: Lesson 5: Implementing Our ML Model

The stage, RAW_SONGS, and processing the lyric PDFs.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `SXGA2_LLM_ADMIN` (ACCOUNTADMIN), `SXGA2_LLM_ROLE` (SYSADMIN) and `SXGA2_LLM_WH` (COMPUTE_WH).
- Course code that no longer works runs **as written** and reports its error; a **Substitute** cell with current functions follows it.
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- 🛠️ **Course update needed** marks what has to change in the course itself.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: Converting the Manual Training to Code

> **Course:** 🥋 LAB: Click the Train Model Button (15 to 45 minutes) (LLMM_161, 162). This turns the manual training into code applied to new PDFs.

> 🛠️ **Course update needed:** There is no Train Model button and no wait. With AI_EXTRACT the extraction is available immediately, so the "proceed to the next page while you wait" framing goes away.

## Page: Test a File Against the Published Model

> **Course:** 🥋 LAB: Create a Stage for Your PDFs (LLMM_164, 165): stage **lyrics_pdfs** in DOC_AI_DB.DOC_AI_SCHEMA, directory table on, server-side encryption, created as DOC_AI_ROLE.

In [ ]:
%%sql -r create_lyrics_pdfs_stage
-- SQL equivalent of the Create Stage dialog (LLMM_165)
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
create stage lyrics_pdfs
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');

> **Course:** 🥋 LAB: Load a Lyrics PDF File Into Your Stage (The Girl From Ipanema by Amy Winehouse.pdf; LLMM_167).

In [ ]:
%%sql -r load_ipanema
-- SQL equivalent of **+ Files**
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
copy files into @doc_ai_db.doc_ai_schema.lyrics_pdfs
from @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
files = ('The Girl From Ipanema by Amy Winehouse.pdf');
alter stage lyrics_pdfs refresh;
list @lyrics_pdfs;

> **Course:** 🥋 LAB: Run Some of the Code From the Model (LLMM_168, 169).

> ⚠️ **Audit note:** The course's code reads `@LYRICS_PDF`, but the stage it just had the learner create is `lyrics_pdfs` (LLMM_165, and the screenshot LLMM_168 uses `@LYRICS_PDFS`). It also calls the model `LYRICS_EXTRACTOR!PREDICT`, which does not exist. It runs as written and reports the error.

In [ ]:
%%sql -r predict_as_given
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
-- the course's block
execute immediate 'begin
  execute immediate ''SELECT DOC_AI_DB.DOC_AI_SCHEMA.LYRICS_EXTRACTOR!PREDICT
(
  GET_PRESIGNED_URL(@LYRICS_PDF, ''''The Girl From Ipanema by Amy Winehouse.pdf''''), 1
)'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r extract_ipanema
-- Substitute: the LYRICS_EXTRACTOR function (AI_EXTRACT + AI_PARSE_DOCUMENT) in place of LYRICS_EXTRACTOR!PREDICT
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
select lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.lyrics_pdfs', 'The Girl From Ipanema by Amy Winehouse.pdf')) as extracted;

## Page: Preparing to Store the Model Output

> **Course:** 🎯 CHALLENGE LAB: Create a New Schema & Award Additional Privileges: USAGE for DOC_AI_ROLE on the SXGA2 database; new schema LYRICS in SXGA2 owned by SYSADMIN; USAGE on LYRICS for DOC_AI_ROLE; table RAW_SONGS in LYRICS with a single column JSON of type VARIANT, owned by SYSADMIN; INSERT and SELECT on RAW_SONGS for DOC_AI_ROLE (LLMM_171, 172).

In [ ]:
%%sql -r create_lyrics_schema
-- Derived solution
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
grant usage on database SXGA2 to role doc_ai_role;
create schema SXGA2.LYRICS;
grant usage on schema SXGA2.LYRICS to role doc_ai_role;
create table SXGA2.LYRICS.RAW_SONGS (json variant);
grant insert, select on table SXGA2.LYRICS.RAW_SONGS to role doc_ai_role;

## Page: Process Files and Store the Output

> **Course:** 🥋 LAB: Add an Insert Command to the Select (LLMM_175), then view the inserted row (LLMM_176).

In [ ]:
%%sql -r insert_ipanema
-- Substitute for the course's `insert into sxga2.lyrics.raw_songs SELECT ...LYRICS_EXTRACTOR!PREDICT(...)`
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
insert into sxga2.lyrics.raw_songs
select lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.lyrics_pdfs', 'The Girl From Ipanema by Amy Winehouse.pdf'));

In [ ]:
%%sql -r view_inserted_row
select * from sxga2.lyrics.raw_songs;

> **Course:** 🎯 CHALLENGE LAB: Load and Process Three More Files (99 Red Balloons, Ribbon in the Sky, Flashdance... What a Feeling). Insert the results into your RAW lyrics table; a select should show 4 rows total.

In [ ]:
%%sql -r load_three_more
-- SQL equivalent of **+ Files**
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
copy files into @doc_ai_db.doc_ai_schema.lyrics_pdfs
from @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
files = ('99 Red Balloons by Nena.pdf', 'Ribbon in the Sky by Stevie Wonder.pdf', 'Flashdance... What a Feeling by Irene Cara.pdf');
alter stage lyrics_pdfs refresh;

In [ ]:
%%sql -r insert_three_more
-- Derived solution: the same insert for each file
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
insert into sxga2.lyrics.raw_songs
select lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.lyrics_pdfs', relative_path))
from directory(@lyrics_pdfs)
where relative_path in ('99 Red Balloons by Nena.pdf', 'Ribbon in the Sky by Stevie Wonder.pdf', 'Flashdance... What a Feeling by Irene Cara.pdf');

In [ ]:
%%sql -r raw_songs_count
select count(*) as rows_in_raw_songs from sxga2.lyrics.raw_songs;

## Page: Parse the Stored JSON Song Data

> **Course:** 🥋 LAB: Parse the JSON Back into the Values we Defined. Run the select below (it parses the 7 value/questions), then the second select that splits each value's confidence score.

> ⚠️ **Audit note:** The course's selects read Document AI's shape (`json:song_title[0].value`, `.score`, `__documentMetadata:ocrScore`). AI_EXTRACT stores `{"error":null,"response":{"song_title":"...", ...}}`, so the course's selects run as written and return NULLs. The substitute reads the new shape and has no score columns.

In [ ]:
%%sql -r parse_json_as_given_1
  select json:__documentMetadata
    , json:song_title
    , json:performing_artist
    , json:lyrics
    , json:authors
    , json:lyrics_copyright_holders
    , json:lyric_licensor_provider
    , json:source
  from sxga2.lyrics.raw_songs;

In [ ]:
%%sql -r parse_json_as_given_2
select json:__documentMetadata:ocrScore as overall_confidence_score
    , json:song_title[0].score as song_title_score
    , json:song_title[0].value::text as song_title
    , json:performing_artist[0].score as perf_art_score
    , json:performing_artist[0].value::text as performing_artist
    , json:lyrics[0].score as lyrics_score
    , json:lyrics[0].value::text as song_lyrics
    , json:authors[0].score as authors_score
    , json:authors[0].value::text as authors
    , json:lyrics_copyright_holders[0].score as copyright_score
    , json:lyrics_copyright_holders[0].value::text as copyright_holders
    , json:lyric_licensor_provider[0].score as provider_score
    , json:lyric_licensor_provider[0].value::text as licensor_provider
    , json:source[0].score as source_score
    , json:source[0].value::text as source
  from sxga2.lyrics.raw_songs;

In [ ]:
%%sql -r parse_json_substitute
-- Substitute: the same seven values from the AI_EXTRACT output (no confidence scores exist)
select json:response:song_title::text as song_title
    , json:response:performing_artist::text as performing_artist
    , json:response:lyrics::text as song_lyrics
    , json:response:authors::text as authors
    , json:response:lyrics_copyright_holders::text as copyright_holders
    , json:response:lyric_licensor_provider::text as licensor_provider
    , json:response:source::text as source
from sxga2.lyrics.raw_songs;

> 🛠️ **Course update needed:** Rewrite this page for the AI_EXTRACT output shape and remove the confidence-score discussion and LLMM_178. The 'stream and task pipeline' pointer refers to a Document AI tutorial; the equivalent now is a stream on the stage directory table with a task that calls AI_EXTRACT.